# 🔮 Step 4: Prediction & Analysis (2024)

Apply the trained 2025 model to 2024 data for classification.

## What This Notebook Does:
- ✅ Load existing trained CNN model (from 2025 training)
- ✅ Load 2024 processed satellite image
- ✅ Make predictions on 2024 data
- ✅ Generate classification map
- ✅ Calculate area changes (km²)
- ✅ Save results for comparison

---

**Previous:** [02_preprocessing_2024.ipynb](02_preprocessing_2024.ipynb)  
**Next:** [05_visualization_reports.ipynb](05_visualization_reports.ipynb)  
**Note:** We skip step 03 (model training) because we use the existing 2025 trained model

## Load Trained Model

In [ ]:
import rasterio
import numpy as np
import joblib
import json
import matplotlib.pyplot as plt
import pandas as pd
from matplotlib.colors import ListedColormap
import matplotlib.patches as mpatches
from scipy.ndimage import generic_filter
import warnings
warnings.filterwarnings('ignore')

print("📂 Loading trained model from 2025 training...")

# Load the trained model and scaler
model = joblib.load("outputs/coastal_classifier_model.pkl")
scaler = joblib.load("outputs/feature_scaler.pkl")

# Load metadata
with open("outputs/model_metadata.json") as f:
    metadata = json.load(f)

print(f"✅ Model loaded successfully!")
print(f"   Type: {type(model).__name__}")
print(f"   Test Accuracy (on 2025 data): {metadata['test_accuracy']*100:.2f}%")
print(f"   Classes: {metadata['classes']}")
print(f"   Training samples: {metadata['training_samples']:,}")

## Load 2024 Processed Image

In [ ]:
# Load the 2024 processed image with all indices
print("📂 Loading 2024 processed image...")

with rasterio.open("processed_image_with_indices_2024.tif") as src:
    image_data_2024 = src.read()
    profile_2024 = src.profile.copy()

bands, height, width = image_data_2024.shape
print(f"✅ Image loaded!")
print(f"   Shape: {image_data_2024.shape} (7 bands, {height}x{width} pixels)")
print(f"   Total pixels: {height * width:,}")

# Check for nodata
nodata_mask = np.all(image_data_2024 == 0, axis=0) | np.any(np.isnan(image_data_2024), axis=0)
valid_pixels = np.sum(~nodata_mask)
print(f"   Valid pixels: {valid_pixels:,}")
print(f"   Nodata pixels: {np.sum(nodata_mask):,}")

## Make Predictions

In [ ]:
# Reshape: (7, H, W) -> (H*W, 7)
print("🔮 Preparing data for prediction...")
reshaped_img = image_data_2024.reshape(bands, -1).T

# Handle NaNs
reshaped_img = np.nan_to_num(reshaped_img, nan=0)

# Scale using the saved scaler (same as training)
reshaped_img_scaled = scaler.transform(reshaped_img)

print(f"   Data shape: {reshaped_img_scaled.shape}")
print(f"\n🔮 Making predictions on 2024 data (this may take a moment)...")

# Predict!
prediction_flat_2024 = model.predict(reshaped_img_scaled)

# Set nodata pixels to 0
prediction_flat_2024[nodata_mask.flatten()] = 0

# Reshape back to map: (H*W) -> (H, W)
prediction_map_2024 = prediction_flat_2024.reshape(height, width)

print(f"✅ Predictions complete!")
unique_classes = np.unique(prediction_flat_2024[prediction_flat_2024 > 0]).astype(int)
print(f"   Predicted classes: {unique_classes}")
print(f"   Class counts: {np.bincount(prediction_flat_2024.astype(int))[1:]}")

## Apply Majority Filter & Save

In [ ]:
# CLASS DEFINITIONS - must match 2025 training data classes
CLASS_INFO = {
    1: {"name": "Seagrass",  "color": "#2ecc71"},  # green
    2: {"name": "Sand",      "color": "#f39c12"},  # orange
    3: {"name": "Seaweed",   "color": "#8e44ad"},  # purple
    4: {"name": "Water",     "color": "#2980b9"},  # blue
    5: {"name": "Landmass",  "color": "#7f8c8d"},  # grey
}

# Apply majority filter to clean salt-and-pepper noise
print("🧹 Applying majority filter to clean noise...")

def majority(values):
    vals = values[values > 0]
    if len(vals) == 0:
        return 0
    counts = np.bincount(vals.astype(int))
    return np.argmax(counts)

prediction_map_clean = generic_filter(
    prediction_map_2024.astype(float),
    majority,
    size=3
).astype(np.uint8)

print("✅ Noise cleaning complete!")

# Save GeoTIFF
profile_2024.update(count=1, dtype=rasterio.uint8, nodata=0)

with rasterio.open("final_classification_map_2024.tif", 'w', **profile_2024) as dst:
    dst.write(prediction_map_clean, 1)

print(f"✅ Classification map saved: 'final_classification_map_2024.tif'")

## Visualize Results

In [ ]:
# Build colormap
unique_classes_clean = sorted([c for c in np.unique(prediction_map_clean) if c > 0])
max_class = max(CLASS_INFO.keys())
colors = ['#ffffff']  # index 0 = nodata = white

for i in range(1, max_class + 1):
    if i in CLASS_INFO:
        colors.append(CLASS_INFO[i]["color"])
    else:
        colors.append('#ffffff')

cmap = ListedColormap(colors)

# Visualize
fig, ax = plt.subplots(figsize=(14, 10))

im = ax.imshow(
    prediction_map_clean,
    cmap=cmap,
    vmin=0,
    vmax=max_class,
    interpolation='nearest'
)

# Legend
legend_patches = []
for cls_id in unique_classes_clean:
    if cls_id in CLASS_INFO:
        info = CLASS_INFO[cls_id]
        pixel_count = np.sum(prediction_map_clean == cls_id)
        total_valid = np.sum(prediction_map_clean > 0)
        pct = (pixel_count / total_valid) * 100 if total_valid > 0 else 0
        label = f"Class {cls_id}: {info['name']} ({pct:.1f}%)"
        patch = mpatches.Patch(color=info['color'], label=label)
        legend_patches.append(patch)

ax.legend(
    handles=legend_patches,
    loc='lower right',
    fontsize=10,
    framealpha=0.9,
    title="Land Cover Classes"
)

ax.set_title("2024 Coastal Assessment Map\n(Using 2025 Trained Model)", 
             fontsize=14, fontweight='bold')
ax.set_xlabel("Pixel Column", fontsize=11)
ax.set_ylabel("Pixel Row", fontsize=11)

plt.tight_layout()
plt.savefig("outputs/final_classification_map_2024.png", dpi=300, bbox_inches='tight')
plt.show()

print("💾 Visualization saved: 'outputs/final_classification_map_2024.png'")

## Calculate Area by Class

In [ ]:
# Define Pixel Size for Sentinel-2
pixel_size_meters = 10
area_per_pixel_sqm = pixel_size_meters * pixel_size_meters  # 100 m²

# Count pixels for each class
unique_labels, counts = np.unique(prediction_flat_2024, return_counts=True)

# Create summary table
results_2024 = []

for label, count in zip(unique_labels, counts):
    if label == 0:
        continue  # Skip nodata
    
    area_sqm = count * area_per_pixel_sqm
    area_hectares = area_sqm / 10000
    area_sqkm = area_sqm / 1e6
    
    class_name = CLASS_INFO.get(int(label), {}).get('name', f'Unknown (Class {label})')
    
    results_2024.append({
        "Year": "2024",
        "Class": int(label),
        "Class Name": class_name,
        "Pixel Count": int(count),
        "Area (m²)": f"{area_sqm:,.0f}",
        "Area (Hectares)": f"{area_hectares:,.2f}",
        "Area (km²)": f"{area_sqkm:,.4f}"
    })

# Create DataFrame
area_df_2024 = pd.DataFrame(results_2024)

print("\n" + "="*80)
print("  📊 2024 COASTAL ASSESSMENT - AREA REPORT")
print("="*80)
print(area_df_2024.to_string(index=False))

# Save report
area_df_2024.to_csv("final_area_report_2024.csv", index=False)
print(f"\n💾 Results saved to 'final_area_report_2024.csv'")

## Summary

In [ ]:
print("\n" + "="*80)
print("  ✅ 2024 PREDICTION & ANALYSIS COMPLETE")
print("="*80)
print(f"\n📂 Generated Files:")
print(f"   • final_classification_map_2024.tif")
print(f"   • final_classification_map_2024.png")
print(f"   • final_area_report_2024.csv")
print(f"\n📊 Model Details:")
print(f"   • Trained on: 2025 data (from notebook 03)")
print(f"   • Applied to: 2024 data (this notebook)")
print(f"   • Accuracy on 2025 training data: {metadata['test_accuracy']*100:.2f}%")
print(f"   • Classes detected: {list(map(int, unique_classes_clean))}")
print(f"\n🔄 Next Step: Open 05_visualization_reports.ipynb")
print(f"   → Will compare 2025 vs 2024 results")
print(f"   → Generate multi-year trend analysis")
print(f"   → Calculate change metrics")
print("="*80)